# Notebook 8 - Run 4 Results Analysis

Evaluates Run 4 results against the Run 3 rerun baseline (62.8% L3).
New in Run 4 vs Run 3: Pass 1 DynamoDB lookup on merchant_name (pre-LLM) and
Pass 2 LLM NER + DynamoDB tool call.

**Input:** `outputs/run4_results_20260312_0631.txt` (CSV format)

**Sections:**
1. Setup and data load
2. Headline metrics
3. L1 / L2a / L3 accuracy vs Run 3 baseline
4. Naive vs pipeline lift
5. Pass 1 DB lookup - coverage and accuracy
6. Pass 2 DB lookup - coverage and accuracy
7. Execution path breakdown
8. Signal conflict analysis (flag field)
9. Confidence vs accuracy
10. Per-category accuracy
11. Per-provider accuracy
12. Top mismatches
13. Ad-hoc query helpers

---
## 1. Setup and Data Load

In [ ]:
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 80)

OUTPUT_DIR = Path('outputs')
ASSET_DIR  = Path('assets')

RUN4_FILE     = OUTPUT_DIR / 'run4_results_20260312_0631.csv'
RUN3_FILE     = OUTPUT_DIR / 'run3_results_20260309_0709.csv'
CAT_KEYS_FILE = ASSET_DIR  / 'cat_keys.csv'
TAXONOMY_FILE = ASSET_DIR / 'taxonomy_master_updated_override.csv'

# Run 3 baseline figures (confirmed)
RUN3_L1  = 85.1
RUN3_L2A = 87.9
RUN3_L3  = 62.8
RUN3_LIFT = 6.2

print('Imports OK')

In [ ]:
# Load Run 4 results
df = pd.read_csv(RUN4_FILE)
print(f'Loaded {len(df):,} rows, {len(df.columns)} columns')
print(df.columns.tolist())

In [ ]:
# Load valid keys
cat_keys_df = pd.read_csv(CAT_KEYS_FILE)
# Handle either column name variant
key_col  = 'base_category_key' if 'base_category_key' in cat_keys_df.columns else 'key'
VALID_KEYS = set(cat_keys_df[key_col].dropna().str.strip())
print(f'Valid keys loaded: {len(VALID_KEYS)}')

# Load taxonomy for spend_type lookups
tax = pd.read_csv(TAXONOMY_FILE)
tax.columns = tax.columns.str.strip()
# Deduplicate mixed keys - keep first (manual_spend_override already applied)
tax = tax.drop_duplicates(subset='base_category_key', keep='first')
print(f'Taxonomy loaded: {len(tax)} keys')

In [ ]:
# Normalise column types
df['confidence']  = pd.to_numeric(df['confidence'], errors='coerce')
df['parse_error'] = df['parse_error'].fillna(False).astype(bool)
df['invalid_key'] = df['invalid_key'].fillna(False).astype(bool)

# Enforce invalid_key from VALID_KEYS (belt-and-braces, mirrors Run 3 notebook)
pred = df['base_category_key'].fillna('')
df['invalid_key'] = (~pred.isin(VALID_KEYS)) & (~df['parse_error'])

# Parse flag column to dict for later analysis
def parse_flag(v):
    if pd.isna(v) or v in ('', 'null', 'None'): return None
    try:    return json.loads(v)
    except: return None

df['flag_parsed']  = df['flag'].apply(parse_flag)
df['flag_type']    = df['flag_parsed'].apply(lambda x: x.get('type') if x else None)
df['flag_detail']  = df['flag_parsed'].apply(lambda x: x.get('detail') if x else None)

# Evaluable subset (no parse error, valid key)
df_eval = df[~df['parse_error'] & ~df['invalid_key']].copy()
print(f'Evaluable rows: {len(df_eval):,} / {len(df):,}')

---
## 2. Headline Metrics

In [ ]:
n           = len(df)
n_parse_err = df['parse_error'].sum()
n_invalid   = df['invalid_key'].sum()
n_eval      = len(df_eval)

# L1 - direct GT label from prop_ideas_has_merch
df_l1  = df_eval[df_eval['gt_spend_type'].isin(['spend', 'non_spend']) & df_eval['spend_type'].notna()]
l1_acc = (df_l1['spend_type'] == df_l1['gt_spend_type']).mean() * 100

# L2a - direct GT label, non-spend only
df_l2a  = df_eval[
    df_eval['gt_category_type'].notna() &
    (df_eval['gt_category_type'] != '') &
    df_eval['category_type'].notna()
]
l2a_acc = (df_l2a['category_type'] == df_l2a['gt_category_type']).mean() * 100 if len(df_l2a) else 0

# L3
df_l3   = df_eval.dropna(subset=['gt_base_category_key', 'base_category_key'])
l3_acc  = (df_l3['base_category_key'] == df_l3['gt_base_category_key']).mean() * 100

# Naive vs pipeline lift
df_lift      = df_eval.dropna(subset=['gt_base_category_key', 'base_category_key', 'base_category_key_naive'])
pipeline_l3  = (df_lift['base_category_key']       == df_lift['gt_base_category_key']).mean() * 100
naive_l3     = (df_lift['base_category_key_naive']  == df_lift['gt_base_category_key']).mean() * 100
lift         = pipeline_l3 - naive_l3

print('=' * 55)
print('RUN 4 RESULTS SUMMARY')
print('=' * 55)
print(f'Total rows:            {n:,}')
print(f'Parse errors:          {n_parse_err} ({n_parse_err/n*100:.1f}%)')
print(f'Invalid keys:          {n_invalid} ({n_invalid/n*100:.1f}%)')
print(f'Evaluable rows:        {n_eval:,} ({n_eval/n*100:.1f}%)')
print()
print(f'L1 spend_type:         {l1_acc:.1f}%  (Run 3: {RUN3_L1}% - NOTE: GT method differs, not directly comparable)  n={len(df_l1)}')
print(f'L2a category_type:     {l2a_acc:.1f}%  (Run 3: {RUN3_L2A}% - NOTE: GT method differs, not directly comparable)  n={len(df_l2a)}')
print(f'L3 exact key:          {l3_acc:.1f}%  (Run 3: {RUN3_L3}%  delta: {l3_acc-RUN3_L3:+.1f}pp - comparable)  n={len(df_l3)}')
print()
print(f'Naive L3:              {naive_l3:.1f}%')
print(f'Pipeline lift:         {lift:+.1f}pp  (Run 3: +{RUN3_LIFT}pp)')

---
## 3. L1 / L2a / L3 vs Run 3 Baseline - Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

metrics  = ['L1 spend_type', 'L2a category_type', 'L3 exact key']
run3_vals = [RUN3_L1, RUN3_L2A, RUN3_L3]
run4_vals = [l1_acc,  l2a_acc,  l3_acc]

x     = np.arange(len(metrics))
width = 0.35
bars3 = ax.bar(x - width/2, run3_vals, width, label='Run 3 baseline', color='#95a5a6')
bars4 = ax.bar(x + width/2, run4_vals, width, label='Run 4',          color='#2ecc71')

for bar in bars3 + bars4:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105)
ax.set_title('Run 4 vs Run 3 baseline - L1 / L2a / L3', fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb8_accuracy_vs_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Naive vs Pipeline Lift

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left - pipeline vs naive bar
axes[0].bar(['Naive', 'Pipeline'], [naive_l3, pipeline_l3],
            color=['#e67e22', '#2ecc71'], width=0.5)
for i, v in enumerate([naive_l3, pipeline_l3]):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10)
axes[0].set_ylabel('L3 Accuracy (%)')
axes[0].set_ylim(0, 100)
axes[0].set_title(f'Pipeline lift: {lift:+.1f}pp  (Run 3: +{RUN3_LIFT}pp)', fontweight='bold')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))

# Right - result breakdown (correct / wrong valid / invalid / parse error)
n_correct  = (df_l3['base_category_key'] == df_l3['gt_base_category_key']).sum()
n_wrong    = len(df_l3) - n_correct
cats  = ['Correct', 'Wrong (valid key)', 'Invalid key', 'Parse error']
vals  = [n_correct, n_wrong, n_invalid, n_parse_err]
clrs  = ['#2ecc71', '#e67e22', '#e74c3c', '#9b59b6']
axes[1].barh(cats, vals, color=clrs, height=0.5)
for i, v in enumerate(vals):
    axes[1].text(v + 2, i, f'{v:,}  ({v/n*100:.1f}%)', va='center', fontsize=9)
axes[1].set_xlim(0, n * 1.3)
axes[1].set_title('Result breakdown', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb8_lift_and_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Pass 1 DB Lookup - Coverage and Accuracy

Pass 1 fires when `merchant_name` field is populated. A hit means `p1_db_category` is not null and not 'NO_MATCH'.

In [ ]:
# Pass 1 coverage
# merchant_name populated = Pass 1 attempted
df_p1_attempted = df[df['merchant_name'].notna() & (df['merchant_name'].str.strip() != '')]
df_p1_hit       = df[df['p1_db_category'].notna() & (df['p1_db_category'] != 'NO_MATCH')]
df_p1_miss      = df[df['p1_db_category'] == 'NO_MATCH']

p1_attempt_rate = len(df_p1_attempted) / len(df) * 100
p1_hit_rate     = len(df_p1_hit) / len(df_p1_attempted) * 100 if len(df_p1_attempted) else 0

# Pass 1 accuracy - where it hit, was the category correct?
df_p1_eval  = df_p1_hit.dropna(subset=['gt_base_category_key'])
p1_acc      = (df_p1_eval['p1_db_category'] == df_p1_eval['gt_base_category_key']).mean() * 100 if len(df_p1_eval) else 0

# Did the final LLM prediction agree with Pass 1 on hit rows?
df_p1_agree = df_p1_hit.dropna(subset=['base_category_key'])
p1_llm_agree = (df_p1_agree['base_category_key'] == df_p1_agree['p1_db_category']).mean() * 100 if len(df_p1_agree) else 0

print('PASS 1 - DynamoDB lookup on merchant_name')
print('-' * 45)
print(f'merchant_name populated (P1 attempted): {len(df_p1_attempted):,}  ({p1_attempt_rate:.1f}% of all rows)')
print(f'P1 hit (DB match found):                {len(df_p1_hit):,}  ({p1_hit_rate:.1f}% of attempted)')
print(f'P1 miss (NO_MATCH):                     {len(df_p1_miss):,}')
print(f'P1 DB category accuracy vs GT:          {p1_acc:.1f}%  (n={len(df_p1_eval)})')
print(f'LLM agreed with P1 hint:                {p1_llm_agree:.1f}%  (n={len(df_p1_agree)})')

In [ ]:
# When LLM overrode Pass 1, was it right or wrong?
df_p1_override = df_p1_agree[
    df_p1_agree['base_category_key'] != df_p1_agree['p1_db_category']
].dropna(subset=['gt_base_category_key'])

if len(df_p1_override):
    # LLM override correct = LLM matched GT; DB was wrong
    llm_right_db_wrong = (df_p1_override['base_category_key'] == df_p1_override['gt_base_category_key']).sum()
    db_right_llm_wrong  = (df_p1_override['p1_db_category']   == df_p1_override['gt_base_category_key']).sum()
    both_wrong          = len(df_p1_override) - llm_right_db_wrong - db_right_llm_wrong
    print(f'\nLLM overrode P1 hint on {len(df_p1_override)} rows:')
    print(f'  LLM correct, DB wrong: {llm_right_db_wrong}  ({llm_right_db_wrong/len(df_p1_override)*100:.1f}%)')
    print(f'  DB correct, LLM wrong: {db_right_llm_wrong}  ({db_right_llm_wrong/len(df_p1_override)*100:.1f}%)')
    print(f'  Both wrong:            {both_wrong}')
    print()
    # Show examples where DB was correct and LLM overrode it wrongly
    override_errors = df_p1_override[
        df_p1_override['p1_db_category'] == df_p1_override['gt_base_category_key']
    ][['original_description', 'merchant_name', 'p1_db_category', 'base_category_key', 'gt_base_category_key', 'reason']]
    print('Sample rows where DB was right and LLM overrode incorrectly:')
    print(override_errors.head(10).to_string(index=False))

---
## 6. Pass 2 DB Lookup - Coverage and Accuracy

Pass 2 fires when LLM NER extracts a merchant name from the description. Hit = `p2_db_category` not null and not 'NO_MATCH'.

In [ ]:
df_p2_ner_ran   = df[df['p2_llm_merchant'].notna()]
df_p2_hit       = df[df['p2_db_category'].notna() & (df['p2_db_category'] != 'NO_MATCH')]
df_p2_miss      = df[df['p2_db_category'] == 'NO_MATCH']

p2_ner_rate     = len(df_p2_ner_ran) / len(df) * 100
p2_hit_rate     = len(df_p2_hit) / len(df_p2_ner_ran) * 100 if len(df_p2_ner_ran) else 0

df_p2_eval      = df_p2_hit.dropna(subset=['gt_base_category_key'])
p2_acc          = (df_p2_eval['p2_db_category'] == df_p2_eval['gt_base_category_key']).mean() * 100 if len(df_p2_eval) else 0

df_p2_agree     = df_p2_hit.dropna(subset=['base_category_key'])
p2_llm_agree    = (df_p2_agree['base_category_key'] == df_p2_agree['p2_db_category']).mean() * 100 if len(df_p2_agree) else 0

print('PASS 2 - LLM NER + DynamoDB tool call')
print('-' * 45)
print(f'NER ran (p2_llm_merchant extracted):    {len(df_p2_ner_ran):,}  ({p2_ner_rate:.1f}% of all rows)')
print(f'P2 hit (DB match found):                {len(df_p2_hit):,}  ({p2_hit_rate:.1f}% of NER attempts)')
print(f'P2 miss (NO_MATCH or NER null):         {len(df_p2_miss):,}')
print(f'P2 DB category accuracy vs GT:          {p2_acc:.1f}%  (n={len(df_p2_eval)})')
print(f'LLM agreed with P2 hint:                {p2_llm_agree:.1f}%  (n={len(df_p2_agree)})')

In [ ]:
# P1 vs P2 combined coverage (rows where at least one pass hit)
p1_hit_mask = df['p1_db_category'].notna() & (df['p1_db_category'] != 'NO_MATCH')
p2_hit_mask = df['p2_db_category'].notna() & (df['p2_db_category'] != 'NO_MATCH')

either_hit   = (p1_hit_mask | p2_hit_mask).sum()
p1_only      = (p1_hit_mask & ~p2_hit_mask).sum()
p2_only      = (~p1_hit_mask & p2_hit_mask).sum()
both_hit     = (p1_hit_mask & p2_hit_mask).sum()
neither      = (~p1_hit_mask & ~p2_hit_mask).sum()

print(f'Combined DB coverage (P1 or P2 hit):    {either_hit:,}  ({either_hit/len(df)*100:.1f}%)')
print(f'  P1 only:   {p1_only:,}')
print(f'  P2 only:   {p2_only:,}  <- P2 incremental value')
print(f'  Both hit:  {both_hit:,}')
print(f'  Neither:   {neither:,}  ({neither/len(df)*100:.1f}%)')

---
## 7. Execution Path Breakdown

Which pipeline steps fired on each row? Shows how often each signal contributed.

In [ ]:
# Execution path frequencies
path_counts = df['execution_path'].value_counts().head(20)
print('Top 20 execution paths:')
print(path_counts.to_string())
print()

# Step coverage - which steps appear in any execution path
STEPS = ['biller', 'p1_db', 'mcc', 'dt', 'merchant', 'regex', 'p2_db', 'llm']
for step in STEPS:
    n_rows = df['execution_path'].fillna('').str.contains(step).sum()
    print(f'  {step:<12}: {n_rows:,}  ({n_rows/len(df)*100:.1f}%)')

In [ ]:
# L3 accuracy by whether P1 or P2 fired
df_ep = df_eval.dropna(subset=['gt_base_category_key', 'base_category_key']).copy()
df_ep['l3_correct'] = df_ep['base_category_key'] == df_ep['gt_base_category_key']

p1_mask = df_ep['p1_db_category'].notna() & (df_ep['p1_db_category'] != 'NO_MATCH')
p2_mask = df_ep['p2_db_category'].notna() & (df_ep['p2_db_category'] != 'NO_MATCH')

groups = {
    'P1 hit (merchant_name DB)': df_ep[p1_mask],
    'P2 hit (NER + DB)':          df_ep[~p1_mask & p2_mask],
    'No DB hit (LLM only)':       df_ep[~p1_mask & ~p2_mask],
}
print('L3 accuracy by DB signal source:')
for label, grp in groups.items():
    if len(grp):
        acc = grp['l3_correct'].mean() * 100
        print(f'  {label:<30}: {acc:.1f}%  (n={len(grp)})')

---
## 8. Signal Conflict Analysis

Conflict = MCC or DT hint disagrees with DB lookup category. LLM raises a `rule_conflict` flag when it detects this. Also check whether the LLM made the right call when signals conflicted.

In [ ]:
# Flag type distribution
flag_dist = df['flag_type'].value_counts(dropna=False)
print('Flag type distribution:')
print(flag_dist.to_string())
print()

# Conflict rows
df_conflict = df[df['flag_type'] == 'rule_conflict'].copy()
print(f'rule_conflict flags: {len(df_conflict):,}  ({len(df_conflict)/len(df)*100:.1f}% of all rows)')

In [ ]:
# For conflict rows: did the LLM make the right call?
if len(df_conflict):
    df_conf_eval = df_conflict.dropna(subset=['gt_base_category_key', 'base_category_key'])
    conf_l3_acc  = (df_conf_eval['base_category_key'] == df_conf_eval['gt_base_category_key']).mean() * 100
    print(f'L3 accuracy on conflict rows: {conf_l3_acc:.1f}%  (n={len(df_conf_eval)})')
    print(f'vs overall L3: {l3_acc:.1f}%')
    print()

    # Sample conflict rows to inspect
    print('Sample conflict rows:')
    cols = ['original_description', 'merchant_name', 'p1_db_category', 'p2_db_category',
            'base_category_key', 'gt_base_category_key', 'flag_detail', 'reason']
    avail = [c for c in cols if c in df_conflict.columns]
    print(df_conflict[avail].head(15).to_string(index=False))

In [ ]:
# Inferred conflicts: DB category != MCC-derived hint (where both present)
# Note: mcc_category_key not saved as a column in Run 4 output; infer from execution_path + reason
# Proxy: rows where p1 or p2 hit AND flag_type is rule_conflict
db_conflict_mask = (
    (p1_hit_mask | p2_hit_mask) &
    (df['flag_type'] == 'rule_conflict')
)
print(f'Rows with DB hit AND rule_conflict flag: {db_conflict_mask.sum()}')
print('These are cases where MCC/DT and DB lookup disagree - inspect LLM decision quality above.')

---
## 9. Confidence vs Accuracy

In [ ]:
df_conf_acc = df_eval.dropna(subset=['confidence', 'base_category_key', 'gt_base_category_key']).copy()
df_conf_acc['l3_match'] = df_conf_acc['base_category_key'] == df_conf_acc['gt_base_category_key']

conf_summary = (
    df_conf_acc.groupby('confidence')
    .agg(count=('l3_match', 'size'), accuracy=('l3_match', 'mean'))
    .reset_index()
)
conf_summary['accuracy'] = (conf_summary['accuracy'] * 100).round(1)
print('Confidence vs L3 accuracy:')
print(conf_summary.to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax2 = ax.twinx()
ax.bar(conf_summary['confidence'], conf_summary['count'], color='#3498db', alpha=0.6, label='Row count')
ax2.plot(conf_summary['confidence'], conf_summary['accuracy'], 'ro-', linewidth=2, markersize=6, label='L3 accuracy')
ax.set_xlabel('Confidence (1-10)')
ax.set_ylabel('Row count')
ax2.set_ylabel('L3 accuracy (%)')
ax2.set_ylim(0, 100)
ax.set_title('Confidence calibration - Run 4', fontweight='bold')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'nb8_confidence_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Per-Category Accuracy

In [ ]:
df_cat = (
    df_l3.groupby('gt_base_category_key')
    .apply(lambda g: pd.Series({
        'total':    len(g),
        'correct':  (g['base_category_key'] == g['gt_base_category_key']).sum(),
        'accuracy': (g['base_category_key'] == g['gt_base_category_key']).mean() * 100,
    }))
    .reset_index()
    .sort_values('accuracy')
)

# Bottom 15 categories
print('Bottom 15 categories by accuracy:')
print(df_cat.head(15).to_string(index=False))
print()
print(f'Perfect (100%) categories: {(df_cat.accuracy == 100).sum()}')
print(f'Zero accuracy categories:  {(df_cat.accuracy == 0).sum()}')

---
## 11. Per-Provider Accuracy

In [ ]:
# provider_name may be in the results if passed through - check
if 'provider_name' in df.columns:
    df_prov = (
        df_l3.groupby('provider_name')
        .apply(lambda g: pd.Series({
            'total':    len(g),
            'correct':  (g['base_category_key'] == g['gt_base_category_key']).sum(),
            'accuracy': (g['base_category_key'] == g['gt_base_category_key']).mean() * 100,
        }))
        .reset_index()
        .sort_values('accuracy')
    )
    print(df_prov.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['#e74c3c' if a < 50 else '#e67e22' if a < 65 else '#2ecc71'
              for a in df_prov['accuracy']]
    ax.barh(df_prov['provider_name'], df_prov['accuracy'], color=colors, height=0.7)
    ax.axvline(l3_acc, color='black', linestyle='--', linewidth=1.2, label=f'Run 4 avg {l3_acc:.1f}%')
    ax.axvline(RUN3_L3, color='grey',  linestyle=':',  linewidth=1.2, label=f'Run 3 avg {RUN3_L3}%')
    ax.set_xlabel('L3 Accuracy (%)')
    ax.set_title('Per-provider L3 accuracy - Run 4', fontweight='bold')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'nb8_provider_accuracy.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('provider_name not in results - skipping provider breakdown')

---
## 12. Top Mismatches

In [ ]:
errors = df_l3[df_l3['base_category_key'] != df_l3['gt_base_category_key']].copy()

top_mismatches = (
    errors.groupby(['gt_base_category_key', 'base_category_key'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)
print('Top 20 GT -> prediction mismatches:')
print(top_mismatches.to_string(index=False))

---
## 13. Ad-Hoc Query Helpers

In [ ]:
# A) All mismatches for a specific GT category
GT_CAT = 'GROCERIES'
(
    errors[errors['gt_base_category_key'] == GT_CAT]
    [['original_description', 'merchant_name', 'p1_db_category', 'p2_db_category',
      'base_category_key', 'gt_base_category_key', 'execution_path', 'reason']]
    .head(20)
)

In [ ]:
# B) All rows where P1 or P2 hit - inspect DB signal quality
(
    df[p1_hit_mask | p2_hit_mask]
    [['original_description', 'merchant_name', 'p1_db_merchant', 'p1_db_category',
      'p2_llm_merchant', 'p2_db_merchant', 'p2_db_category',
      'base_category_key', 'gt_base_category_key', 'execution_path']]
    .head(30)
)

In [ ]:
# C) Rows where NER ran but got no DB hit - what merchant names did the LLM extract?
df_ner_miss = df[df['p2_llm_merchant'].notna() & (df['p2_db_category'] == 'NO_MATCH')]
print(f'NER ran, no DB hit: {len(df_ner_miss)} rows')
print('Sample extracted merchants that did not match:')
print(df_ner_miss['p2_llm_merchant'].value_counts().head(30).to_string())

In [ ]:
# D) Free-text search across descriptions
SEARCH = 'woolworths'
(
    df[df['original_description'].str.lower().str.contains(SEARCH, na=False)]
    [['original_description', 'merchant_name', 'p1_db_category', 'p2_db_category',
      'base_category_key', 'gt_base_category_key', 'execution_path', 'reason']]
    .head(20)
)

In [ ]:
# E) Spend type errors - what is the LLM getting wrong at L1?
l1_errors = df_l1[df_l1['spend_type'] != df_l1['gt_spend_type']]
print(f'L1 errors: {len(l1_errors)}')
(
    l1_errors[['original_description', 'merchant_name', 'spend_type',
               'gt_spend_type', 'base_category_key', 'gt_base_category_key',
               'execution_path', 'reason']]
    .head(20)
)